# NyayaAI - Colab Training Notebook

End-to-end pipeline for **IndiaLex-FT** — a LoRA fine-tuned LLM for Indian legal and income-tax Q&A.

**Steps covered:**
1. Clone repo and install dependencies
2. Load API keys from Colab Secrets
3. Baseline evaluation (pre-training)
4. LoRA SFT fine-tuning
5. Post-training evaluation
6. LLM-as-Judge scoring via claude-sonnet-4-6
7. Download outputs

> **Runtime:** Select `Runtime → Change runtime type → T4 GPU` before starting.

In [1]:
!git clone https://github.com/Harshith2014/Nyaya-AI.git
%cd Nyaya-AI/indialex-ft

Cloning into 'Nyaya-AI'...
remote: Enumerating objects: 109, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 109 (delta 42), reused 108 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (109/109), 38.10 KiB | 4.23 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/Nyaya-AI/indialex-ft


In [2]:
!pip install -r requirements.txt --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 960.7/960.7 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 109.9 MB/s eta 0:00:00


In [3]:
import os
from google.colab import userdata

def _load_secret(name, required=True):
    try:
        val = userdata.get(name)
        os.environ[name] = val
        print(f"{name} loaded: True")
    except Exception:
        if required:
            print(f"{name} loaded: MISSING (required — add it in Secrets)")
        else:
            print(f"{name} loaded: skipped (optional)")

_load_secret("ANTHROPIC_API_KEY", required=True)
_load_secret("GROQ_API_KEY",      required=True)
_load_secret("WANDB_API_KEY",     required=False)
_load_secret("HF_TOKEN",          required=False)

ANTHROPIC_API_KEY loaded: True
GROQ_API_KEY loaded: True
WANDB_API_KEY loaded: skipped (optional)
HF_TOKEN loaded: skipped (optional)


In [4]:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("HuggingFace login successful.")
else:
    print("HF_TOKEN not set — skipping HuggingFace login (only needed for gated models).")

HF_TOKEN not set — skipping HuggingFace login (only needed for gated models).


## Step 1: Baseline Evaluation

Runs inference on the test split using the **base model** (before fine-tuning).
Computes ROUGE-L and BERTScore F1, and saves results to `evals/baseline_results.json`.

In [5]:
!python scripts/baseline_eval.py

INFO  Loading tokenizer and model: meta-llama/Llama-3.2-3B-Instruct
INFO  HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO  HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json "HTTP/1.1 401 Unauthorized"
INFO  HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json "HTTP/1.1 401 Unauthorized"
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 772, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '401 Unauthorized' for url 'https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

## Step 2: Training

Fine-tunes the base model with **LoRA + QLoRA (4-bit NF4)** via TRL SFTTrainer.
Saves the LoRA adapter to `outputs/adapter/` and the merged model to `outputs/merged/`.
Logs metrics to Weights & Biases.

In [6]:
!python scripts/train.py

INFO  NumExpr defaulting to 2 threads.
INFO  TensorFlow version 2.20.0 available.
INFO  JAX version 0.7.2 available.
Traceback (most recent call last):
  File "/content/Nyaya-AI/indialex-ft/scripts/train.py", line 186, in <module>
    main()
  File "/content/Nyaya-AI/indialex-ft/scripts/train.py", line 83, in main
    from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
ImportError: cannot import name 'DataCollatorForCompletionOnlyLM' from 'trl' (/usr/local/lib/python3.12/dist-packages/trl/__init__.py)


## Step 3: Post-training Evaluation

Runs inference on the test split using the **fine-tuned merged model**.
Computes ROUGE-L and BERTScore F1, compares against baseline, and saves
a delta summary to `evals/ft_results.json`.

In [7]:
!python scripts/evaluate.py

WARNING  Baseline file not found at /content/Nyaya-AI/indialex-ft/evals/baseline_results.json — skipping delta comparison.
INFO  Loading fine-tuned model from outputs/indialex-ft/merged
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py", line 437, in cached_files
    hf_hub_download(
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 84, in _inner_fn
    validate_repo_id(arg_value)
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 132, in validate_repo_id
    raise HFValidationError(
huggingface_hub.errors.HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': 'outputs/indialex-ft/merged'. Use `repo_type` argument if needed.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_aut

## Step 4: LLM-as-Judge Evaluation

Uses **claude-sonnet-4-6** (Anthropic API) to score each fine-tuned answer on
four legal-domain dimensions (1–5 each): correctness, faithfulness, helpfulness,
and legal accuracy. Results saved to `evals/judge_results.json`.

In [8]:
!python evals/llm_judge.py --results evals/ft_results.json

ERROR: evals/ft_results.json not found. Run evaluate.py (or baseline_eval.py) first.


In [ ]:
import shutil

shutil.make_archive("nyayaai_outputs", "zip", root_dir=".", base_dir="outputs")
print("Zipped outputs/ → nyayaai_outputs.zip")

shutil.make_archive("nyayaai_evals", "zip", root_dir=".", base_dir="evals")
print("Zipped evals/   → nyayaai_evals.zip")

try:
    from google.colab import files
    files.download("nyayaai_outputs.zip")
    files.download("nyayaai_evals.zip")
except ImportError:
    print("Not running in Colab — skipping auto-download.")
    print("Zip files saved locally: nyayaai_outputs.zip, nyayaai_evals.zip")